# Chapter 4 — Chunking for Scientific Documents (v2026)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IvanReznikov/LangChain4LifeSciencesHealthcare)

**Learning objectives**
- Compare fixed-size, recursive, and structure-aware chunking
- Keep tables, protocols, and sections atomic
- Measure chunk-size distribution and boundary quality
- Choose a strategy per document type

> Runtime: ~2 min (CPU)  
> Cost: $0  
> Data: synthetic paper/protocol text

> **LangChain 1.x (2026)** — built on `langchain-core==1.2.30`, `langchain==1.0.0`. See `UPDATE_2026.md`.


**Chunking is the most under-appreciated RAG decision.** Split a chemical table across two chunks and neither half is retrievable; too-big chunks blow the context budget.

Scientific documents have structure - sections, tables, protocols. Naive splitting ignores it. We compare strategies and keep atomic units intact.


## API keys & credentials

Mostly local (no paid API needed); the bootstrap sets up an optional provider for gated LLM cells.


In [1]:
# @title Setting environmental variables
import os

# --- Dual-mode secrets: works in Google Colab AND locally (.env / environment) ---
try:
    from google.colab import userdata  # type: ignore

    IN_COLAB = True
except Exception:
    userdata = None
    IN_COLAB = False

if not IN_COLAB:
    # Local run: load variables from a .env file if present (never commit .env!).
    try:
        from dotenv import load_dotenv

        load_dotenv()
    except Exception:
        pass


def get_secret(name, default=None):
    """Read a secret from Colab Secrets, else from local env/.env, else default."""
    if IN_COLAB and userdata is not None:
        try:
            val = userdata.get(name)
            if val:
                return val
        except Exception:
            pass
    return os.getenv(name, default)


# 👇 Choose your provider 👇
API_KEY_PROVIDER = "OPENAI"  # "GEMINI" | "OPENAI" | "GROQ" | "ANTHROPIC"

if API_KEY_PROVIDER == "OPENAI":
    os.environ["OPENAI_API_KEY"] = get_secret("LC4LSH_OPENAI_API_KEY", "sk-...")
elif API_KEY_PROVIDER == "ANTHROPIC":
    os.environ["ANTHROPIC_API_KEY"] = get_secret(
        "LC4LSH_ANTHROPIC_API_KEY", "sk-ant-..."
    )
elif API_KEY_PROVIDER == "GEMINI":
    os.environ["GOOGLE_API_KEY"] = get_secret("LC4LSH_GOOGLE_API_KEY", "AIza...")
elif API_KEY_PROVIDER == "GROQ":
    os.environ["GROQ_API_KEY"] = get_secret("LC4LSH_GROQ_API_KEY", "gsk_...")

print(
    f"✅ API keys loaded for {API_KEY_PROVIDER} (source: {'Colab Secrets' if IN_COLAB else 'local env/.env'})"
)

# Hugging Face token (optional; needed for gated models)
os.environ["HF_TOKEN"] = get_secret("HF_TOKEN", "") or ""

API keys loaded for OPENAI


## Installation (pinned)


In [2]:
# @title Installing Python dependencies
%pip install -q "langchain==1.0.0" "langchain-core==1.2.30" "langchain-text-splitters==1.0.0" "tiktoken>=0.7" python-dotenv
# Pinned versions - Last validated: 2026-07-21 (see UPDATE_2026.md)

In [3]:
# @title Setting LangSmith variables
LANGSMITH_API_KEY = get_secret("LANGSMITH_API_KEY", "lsv2_pt_...")
LANGSMITH_PROJECT = "lc4lsh-chapter4-chunking"
REGION = "US"
if LANGSMITH_API_KEY and LANGSMITH_API_KEY.startswith("lsv2_"):
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
    os.environ["LANGSMITH_PROJECT"] = LANGSMITH_PROJECT
    print("LangSmith ON ->", LANGSMITH_PROJECT)
else:
    os.environ["LANGSMITH_TRACING"] = "false"
    print("LangSmith OFF")

LangSmith ON -> lc4lsh-chapter4-chunking


## A synthetic scientific document


In [4]:
DOC = """# Aspirin Synthesis and Characterization

## Abstract
Aspirin (acetylsalicylic acid) is synthesized via esterification of salicylic acid.

## Introduction
Aspirin is a widely used NSAID. Its SMILES is CC(=O)Oc1ccccc1C(=O)O.

## Protocol
1. Add 2.0 g salicylic acid to a flask.
2. Add 5 mL acetic anhydride and 5 drops H2SO4.
3. Heat at 85 C for 15 min.
4. Cool and add ice water to precipitate.

## Results
| Compound | Yield (%) | mp (C) |
|----------|-----------|--------|
| Aspirin  | 78        | 135    |
| Batch-2  | 81        | 136    |

## Conclusion
Pure aspirin was obtained in good yield.
"""
print("Document:", len(DOC), "chars")

Document: 599 chars


## 1. Fixed-size character splitting


In [5]:
from langchain_text_splitters import CharacterTextSplitter

fixed = CharacterTextSplitter(chunk_size=200, chunk_overlap=0, separator=" ")
fixed_chunks = fixed.split_text(DOC)
print("fixed ->", len(fixed_chunks), "chunks")
for c in fixed_chunks[:3]:
    print("---", repr(c[:80]))

fixed -> 3 chunks
--- '# Aspirin Synthesis and Characterization\n\n## Abstract\nAspirin (acetylsalicylic a'
--- 'CC(=O)Oc1ccccc1C(=O)O.\n\n## Protocol\n1. Add 2.0 g salicylic acid to a flask.\n2. A'
--- 'Results\n| Compound | Yield (%) | mp (C) |\n|----------|-----------|--------|\n| As'


## 2. Recursive (paragraph-aware)


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

rec = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=30)
rec_chunks = rec.split_text(DOC)
print("recursive ->", len(rec_chunks), "chunks")

recursive -> 3 chunks


## 3. Structure-aware (keep sections & tables atomic)


In [7]:
import re


def section_split(text):
    parts = re.split(r"(?m)^(#{1,3} .*)$", text)
    out, cur, buf = [], "Preamble", ""
    for p in parts:
        if re.match(r"^#{1,3} ", p):
            if buf.strip():
                out.append({"section": cur.strip("# ").strip(), "text": buf.strip()})
            cur, buf = p, ""
        else:
            buf += p
    if buf.strip():
        out.append({"section": cur.strip("# ").strip(), "text": buf.strip()})
    return out


sec = section_split(DOC)
print("structure ->", len(sec), "section chunks")
for s in sec:
    print("  [", s["section"], "]", len(s["text"]), "chars  table=", "|" in s["text"])

structure -> 5 section chunks
  [ Abstract ] 83 chars  table= False
  [ Introduction ] 68 chars  table= False
  [ Protocol ] 157 chars  table= False
  [ Results ] 135 chars  table= True
  [ Conclusion ] 40 chars  table= False


## 4. Compare distributions & table integrity


In [8]:
import statistics as st


def stats(name, chunks):
    s = [len(c) for c in chunks]
    print(name, "n=", len(s), "mean=", int(st.mean(s)), "min=", min(s), "max=", max(s))


stats("fixed", fixed_chunks)
stats("recursive", rec_chunks)
stats("section", [s["text"] for s in sec])


def table_split(chunks):
    return any(("| Compound" in c) != ("| Batch-2" in c) for c in chunks)


print(
    "table split? fixed=",
    table_split(fixed_chunks),
    "recursive=",
    table_split(rec_chunks),
)

fixed n= 3 mean= 191 min= 177 max= 200
recursive n= 3 mean= 198 min= 169 max= 223
section n= 5 mean= 96 min= 40 max= 157
table split? fixed= False recursive= False


## 5. Token-aware chunking


In [9]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")
tok = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base", chunk_size=120, chunk_overlap=20
)
tok_chunks = tok.split_text(DOC)
print("token-aware ->", len(tok_chunks), "chunks (<=120 tokens)")
print("token counts:", [len(enc.encode(c)) for c in tok_chunks])

token-aware -> 3 chunks (<=120 tokens)
token counts: [72, 112, 12]


## Choosing a strategy

| Document type | Recommended |
|---|---|
| Narrative papers | Recursive |
| Protocols/methods | Structure-aware |
| Tables/spectra | Keep atomic |
| Mixed reports | Structure-aware + token cap |
| Fixed context budget | Token-aware |


## Limitations & safety notes

- **No free lunch.** Small chunks retrieve precisely but lose context; large chunks preserve context but dilute retrieval.
- **Tables/figures.** Text splitters do not understand rendered tables; extract them separately.
- **Overlap != structure.** Overlap helps prose but cannot rescue a table split mid-row.


---
### Further Reading

| Notebook | Relevance |
|----------|-----------|
| **Chapter 4 RAG** | Where chunking meets retrieval |
| **Chapter 4 Scientific Ingestion and Metadata** | Metadata-preserving ingestion pipeline |


In [10]:
# Cleanup
import gc

for _v in ("fixed_chunks", "rec_chunks", "sec", "tok_chunks"):
    globals().pop(_v, None)
gc.collect()
print("Cleanup complete.")

Cleanup complete.


## Exercises

<details><summary>Why is splitting a table across chunks harmful?</summary>Each half loses rows or the header, so neither is independently retrievable or interpretable.</details>

<details><summary>When prefer token-aware over character-aware?</summary>When you must respect a model's token budget precisely.</details>

<details><summary>What metadata should each chunk carry?</summary>doc_id, section, chunk index, source path - for citation back to origin.</details>

### Tasks
- **Task A** - Write `table_aware_split` extracting Markdown tables into their own chunks.
- **Task B** - Sweep chunk_size in {100,250,500,1000} and plot mean count vs size.
- **Task C** - Add chunk-level metadata (doc_id, section, chunk_index) to the section splitter.
- **Task D** - Add a token cap (256) to the structure-aware splitter, splitting long sections recursively.
